In [99]:
import os

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_image
from torchvision.transforms import v2

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [100]:
class SimpsonsDataset(Dataset):
    def __init__(self, path, transform=None):
        self.path = path
        self.transform = transform

        self.classes = sorted(
            [d for d in os.listdir(self.path) if os.path.isdir(os.path.join(self.path, d))]
        )

        self.class_to_idx = {class_name: i for i, class_name in enumerate(self.classes)}

        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(self.path, class_name)

            for file_name in sorted(os.listdir(class_dir)):
                file_path = os.path.join(class_dir, file_name)

                if os.path.isfile(file_path):
                    self.samples.append((file_path, self.class_to_idx[class_name]))

    def __getitem__(self, index):
        image_path, label = self.samples[index]

        img = read_image(str(image_path))   # Tensor [C, H, W], uint8

        if self.transform is not None:
            img = self.transform(img)

        return img, label

    def __len__(self):
        return len(self.samples)

In [101]:
train_transform = v2.Compose([
    v2.RandomResizedCrop((224, 224), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=10),
    v2.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406],
                 std=[0.229, 0.224, 0.225]),
])

val_transform = v2.Compose([
    v2.Resize((224, 224), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406],
                 std=[0.229, 0.224, 0.225]),
])

In [ ]:
import numpy as np
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split

PATH = 'data/simpsons_dataset'
SEED = 42
RARE_THRESHOLD  = 5

train_dataset = SimpsonsDataset(path=PATH, transform=train_transform)
val_dataset = SimpsonsDataset(path=PATH, transform=val_transform)

labels = [label for _, label in train_dataset.samples]
indices = np.arange(len(labels))

class_counts = Counter(labels)
rare_classes = {cls for cls, count in class_counts.items() if count < RARE_THRESHOLD}

rare_indices = []
normal_indices = []

for i, label in enumerate(labels):
    if label in rare_classes:
        rare_indices.append(i)
    else:
        normal_indices.append(i)

normal_labels = [labels[i] for i in normal_indices]

train_idx_norm, temp_idx_norm = train_test_split(
    normal_indices,
    test_size=0.2,
    stratify=normal_labels,
    random_state=SEED
)

temp_labels = [labels[i] for i in temp_idx_norm]

val_idx_norm, test_idx_norm = train_test_split(
    temp_idx_norm,
    test_size=0.5,
    stratify=temp_labels,
    random_state=SEED
)

train_idx = list(train_idx_norm) + rare_indices
val_idx   = list(val_idx_norm)
test_idx  = list(test_idx_norm)

In [103]:
from collections import Counter

print("Train:", Counter([labels[i] for i in train_idx]))
print("Val:",   Counter([labels[i] for i in val_idx]))
print("Test:",  Counter([labels[i] for i in test_idx]))

Train: Counter({15: 1797, 28: 1163, 27: 1162, 20: 1083, 4: 1074, 22: 1033, 17: 965, 32: 955, 6: 954, 25: 863, 7: 789, 0: 730, 37: 702, 2: 498, 16: 398, 9: 375, 11: 366, 29: 286, 18: 248, 24: 197, 42: 145, 21: 102, 14: 97, 3: 85, 36: 82, 5: 78, 35: 71, 31: 58, 23: 57, 33: 52, 40: 44, 8: 38, 34: 36, 1: 33, 38: 32, 30: 26, 13: 22, 12: 22, 26: 14, 10: 6, 41: 6, 19: 3})
Val: Counter({15: 224, 28: 145, 27: 145, 20: 135, 4: 134, 22: 129, 17: 121, 32: 120, 6: 119, 25: 108, 7: 99, 0: 92, 37: 87, 2: 62, 16: 50, 9: 47, 11: 45, 29: 36, 18: 31, 24: 24, 42: 18, 21: 13, 14: 12, 3: 11, 36: 11, 5: 10, 35: 9, 33: 7, 23: 7, 31: 7, 34: 5, 1: 5, 40: 5, 38: 4, 8: 4, 30: 3, 12: 3, 26: 2, 13: 2, 41: 1, 10: 1})
Test: Counter({15: 225, 28: 146, 27: 145, 20: 136, 4: 134, 22: 129, 17: 120, 6: 120, 32: 119, 25: 108, 7: 98, 0: 91, 37: 88, 2: 63, 16: 50, 9: 47, 11: 46, 29: 36, 18: 31, 24: 25, 42: 18, 21: 13, 14: 12, 36: 10, 5: 10, 3: 10, 35: 9, 31: 7, 23: 7, 33: 6, 40: 6, 8: 5, 38: 4, 34: 4, 1: 4, 30: 3, 13: 3, 12: 

In [ ]:
from torch.utils.data import Subset

train_dataset = Subset(train_dataset, train_idx)
val_dataset   = Subset(val_dataset, val_idx)
test_dataset  = Subset(val_dataset, test_idx)